# PokerBots on Chipzen
Use a standard CPU runtime. Run setup and the offline check first. The bundled Value policy is ready for a friendly match, or you can run the optional trainer to select a policy on new deals. Only the final cell connects to Chipzen, and it is off by default.

## 1. Set up the runtime

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

REPO = Path(os.environ.get('POKERBOTS_REPO', '/content/PokerBots'))
if not REPO.is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/Shaurya-S0603/PokerBots.git',
                    str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(REPO / 'requirements-chipzen.txt')], check=True)
POLICY = REPO / 'training' / 'competition-policy.json'
assert POLICY.is_file(), 'Use the PokerBots revision that includes the competition policy.'
print(f"Bundled policy: {json.loads(POLICY.read_text())['selected']}")

## 2. Verify the adapter offline

In [ ]:
subprocess.run([sys.executable, 'tests/chipzen_conformance.py'], cwd=REPO, check=True)

## 3. Train on new deals (optional)
Turn on training to compare candidates over seat-swapped hands. Use a fresh seed for a new experimental round. The trainer keeps the incumbent unless the candidate's held-out interval against it clears zero. Save the JSON to keep your policy after the runtime ends.

In [ ]:
RUN_TRAINING = False # @param {type:"boolean"}
SCREENING_PAIRS = 24 # @param {type:"integer"}
HOLDOUT_PAIRS = 64 # @param {type:"integer"}
SEED = 21000 # @param {type:"integer"}
STACK = 1000 # @param {type:"integer"}
if RUN_TRAINING:
    output = REPO / 'colab-policy.json'
    workers = min(8, len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else (os.cpu_count() or 1))
    subprocess.run([sys.executable, 'competition_train.py', '--pairs', str(SCREENING_PAIRS),
                    '--held-out', str(HOLDOUT_PAIRS), '--seed', str(SEED), '--stack', str(STACK),
                    '--workers', str(workers), '--incumbent', str(POLICY), '--output', str(output)], cwd=REPO, check=True)
    POLICY = output
report = json.loads(POLICY.read_text())
candidate = report['candidate']
print(f"Selected: {report['selected']} | Evaluated: {candidate}")
for opponent, row in report['evaluation'][candidate].items():
    lo, hi = row['interval_95']
    print(f"{opponent:18s} {row['bb_per_100']:+8.2f} bb/100  [{lo:+.2f}, {hi:+.2f}] ({row['pairs']} pairs)")

In [ ]:
DOWNLOAD_POLICY = False # @param {type:"boolean"}
if DOWNLOAD_POLICY:
    from google.colab import files
    files.download(str(POLICY))

## 4. Compare two strategies (optional)
Use the same deals with seats swapped. For your own bot, put a Python module in the checkout and enter its `module:callable` here.

In [ ]:
RUN_MATCHUP = False # @param {type:"boolean"}
CANDIDATE = "Button Pressure" # @param {type:"string"}
CONTROL = "Value" # @param {type:"string"}
MATCHUP_PAIRS = 128 # @param {type:"integer"}
MATCHUP_SEED = 27000 # @param {type:"integer"}
if RUN_MATCHUP:
    subprocess.run([sys.executable, 'strategy_matchup.py', '--candidate', CANDIDATE,
                    '--control', CONTROL, '--pairs', str(MATCHUP_PAIRS),
                    '--seed', str(MATCHUP_SEED)], cwd=REPO, check=True)

## 5. Connect for one friendly match
Create a **remote / external API** heads-up Hold’em bot in Chipzen. In Colab **Secrets**, set `CHIPZEN_BOT_ID` and `CHIPZEN_EXTBOT_TOKEN`, and grant this notebook access. Enable the cell, wait for the lobby connection, then start a friendly challenge in Chipzen. Colab can reset or disconnect. Use a stable host or Chipzen's hosted track for scheduled matches.

In [ ]:
CONNECT = False # @param {type:"boolean"}
ENVIRONMENT = "prod" # @param ["prod", "staging"]
MATCHES = 1 # @param {type:"integer"}
REMOTE_STRATEGY = "" # @param {type:"string"}
if CONNECT:
    from google.colab import userdata
    names = ('CHIPZEN_BOT_ID', 'CHIPZEN_EXTBOT_TOKEN')
    env = {**os.environ, **{name: userdata.get(name) for name in names}}
    selection = ['--strategy', REMOTE_STRATEGY] if REMOTE_STRATEGY else ['--policy', str(POLICY)]
    subprocess.run([sys.executable, 'chipzen_entry.py', *selection,
                    '--env', ENVIRONMENT, '--matches', str(MATCHES)], cwd=REPO, env=env, check=True)